# 04 - DML avançado: UPDATE, DELETE e MERGE

## Objetivo

Demonstrar operações transacionais em Iceberg.

## Valor para a PRODEMGE

Correção de pagamentos, cancelamento de benefícios e carga incremental de sistemas de origem.

## Como usar

1. Substitua os placeholders (`<datahub_link>`, `<usuário>` e `<senha>`) no primeiro bloco de código.
2. Execute a célula de configuração Livy.
3. Execute as células da demo.
4. No final, encerre a sessão Livy.

> Este notebook não executa Spark localmente. Todo código Spark SQL/PySpark é submetido ao Livy3 via REST API.

In [4]:
import requests
import time
import json
import urllib3

urllib3.disable_warnings()

# =====================================================================
# CONFIGURAÇÕES DO LIVY3 - AJUSTE ESTES VALORES
# =====================================================================

LIVY_URL = "<datahub_link>"
USERNAME = "<usuário>"
PASSWORD = "<senha>"

# =====================================================================
# CONFIGURAÇÕES DE SESSÃO SPARK
# =====================================================================

# Se o catálogo iceberg_prod já estiver configurado no cluster,
# você pode remover as configs spark.sql.catalog.* abaixo.
SESSION_CONF = {
    "spark.app.name": "PRODEMGE_Iceberg_Demo_Livy3",
    "spark.executor.memory": "4g",
    "spark.executor.cores": "2",
    "spark.executor.instances": "2",
    "spark.driver.memory": "2g",

    # Catálogo Iceberg. Ajuste conforme o padrão do seu ambiente.
    "spark.sql.catalog.iceberg_prod": "org.apache.iceberg.spark.SparkCatalog",
    "spark.sql.catalog.type": "hive",
    "spark.sql.extensions": "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions"
}

# =====================================================================
# CLIENTE LIVY3
# =====================================================================

http = requests.Session()
http.auth = (USERNAME, PASSWORD)
http.verify = False

HEADERS = {"Content-Type": "application/json"}


def pretty_json(obj):
    """Imprime JSON formatado para facilitar troubleshooting."""
    print(json.dumps(obj, indent=2, ensure_ascii=False))


def create_livy_session(kind="pyspark", conf=None, timeout_seconds=600):
    """
    Cria uma sessão Livy interativa.

    kind="pyspark" permite enviar código PySpark e Spark SQL usando spark.sql(...).
    """
    payload = {
        "kind": kind,
        "conf": conf or SESSION_CONF
    }

    response = http.post(
        f"{LIVY_URL}/sessions",
        headers=HEADERS,
        data=json.dumps(payload)
    )
    response.raise_for_status()

    session_id = response.json()["id"]
    print(f"Sessão Livy criada: {session_id}")

    start = time.time()

    while True:
        response = http.get(f"{LIVY_URL}/sessions/{session_id}")
        response.raise_for_status()

        payload = response.json()
        state = payload.get("state")
        print(f"Estado da sessão: {state}")

        if state == "idle":
            print("Sessão Livy pronta para receber statements.")
            return session_id

        if state in ["dead", "error", "killed"]:
            pretty_json(payload)
            raise RuntimeError(f"Falha ao criar sessão Livy. Estado: {state}")

        if time.time() - start > timeout_seconds:
            raise TimeoutError("Timeout aguardando sessão Livy ficar idle.")

        time.sleep(5)


def submit_statement(session_id, code, kind="pyspark", timeout_seconds=900):
    """
    Submete um statement para uma sessão Livy existente.

    O código enviado deve ser uma string Python válida.
    Para SQL, use spark.sql(\"\"\" ... \"\"\").show().
    """
    payload = {
        "code": code,
        "kind": kind
    }

    response = http.post(
        f"{LIVY_URL}/sessions/{session_id}/statements",
        headers=HEADERS,
        data=json.dumps(payload)
    )
    response.raise_for_status()

    statement_id = response.json()["id"]
    print(f"Statement submetido: {statement_id}")

    start = time.time()

    while True:
        response = http.get(
            f"{LIVY_URL}/sessions/{session_id}/statements/{statement_id}"
        )
        response.raise_for_status()

        result = response.json()
        state = result.get("state")
        print(f"Estado do statement: {state}")

        if state == "available":
            output = result.get("output", {})
            pretty_json(output)
            return output

        if state in ["error", "cancelling", "cancelled"]:
            pretty_json(result)
            raise RuntimeError(f"Statement falhou. Estado: {state}")

        if time.time() - start > timeout_seconds:
            raise TimeoutError("Timeout aguardando statement finalizar.")

        time.sleep(3)


def close_livy_session(session_id):
    """Encerra a sessão Livy para liberar recursos no cluster."""
    response = http.delete(f"{LIVY_URL}/sessions/{session_id}")
    if response.status_code in [200, 202, 204]:
        print(f"Sessão Livy encerrada: {session_id}")
    else:
        print(f"Não foi possível encerrar a sessão {session_id}.")
        print(response.text)


# Cria uma sessão Livy para este notebook.
livy_session_id = create_livy_session()

Sessão Livy criada: 11
Estado da sessão: starting
Estado da sessão: starting
Estado da sessão: starting
Estado da sessão: starting
Estado da sessão: starting
Estado da sessão: starting
Estado da sessão: starting
Estado da sessão: idle
Sessão Livy pronta para receber statements.


In [5]:
# Normaliza o username para usar no nome da tabela sem caracteres inválidos.
table_suffix = "".join(ch if ch.isalnum() or ch == "_" else "_" for ch in USERNAME).strip("_")
table_name = f"beneficios_{table_suffix}"
particionado_table = f"beneficios_particionados_{table_suffix}"
staging_table = f"beneficios_staging_{table_suffix}"
csv_table = f"beneficios_csv_{table_suffix}"

spark_code = f"""
# =====================================================================
# DML AVANÇADO COM ICEBERG
# =====================================================================

# Garante tabela principal.
spark.sql(\"\"\"
CREATE TABLE IF NOT EXISTS governo_mg.{table_name} (
    id_cidadao BIGINT,
    nome STRING,
    valor_beneficio DOUBLE,
    data_pagamento DATE,
    status STRING
)
USING iceberg
\"\"\")

# Insere dados base caso esteja vazia.
spark.sql(\"\"\"
INSERT INTO governo_mg.{table_name}
SELECT *
FROM VALUES
(100, 'Beneficiário A', 400.00, DATE '2025-04-01', 'ATIVO'),
(101, 'Beneficiário B', 550.00, DATE '2025-04-02', 'ATIVO'),
(102, 'Beneficiário C', 300.00, DATE '2025-04-03', 'SUSPENSO')
AS t(id_cidadao, nome, valor_beneficio, data_pagamento, status)
\"\"\")

# UPDATE: corrige valor de benefício.
spark.sql(\"\"\"
UPDATE governo_mg.{table_name}
SET valor_beneficio = 999.99
WHERE id_cidadao = 100
\"\"\")

# DELETE: remove ou cancela registros que não devem mais compor a base.
spark.sql(\"\"\"
DELETE FROM governo_mg.{table_name}
WHERE status = 'SUSPENSO'
\"\"\")

# Cria tabela staging para simular carga incremental.
spark.sql(\"\"\"
DROP TABLE IF EXISTS governo_mg.{staging_table}
\"\"\")

spark.sql(\"\"\"
CREATE TABLE governo_mg.{staging_table} (
    id_cidadao BIGINT,
    nome STRING,
    valor_beneficio DOUBLE,
    data_pagamento DATE,
    status STRING
)
USING iceberg
\"\"\")

spark.sql(\"\"\"
INSERT INTO governo_mg.{staging_table} VALUES
(100, 'Beneficiário A', 1200.00, DATE '2025-05-01', 'ATIVO'),
(200, 'Novo Beneficiário', 300.00, DATE '2025-05-01', 'ATIVO')
\"\"\")

# MERGE: atualiza quem já existe e insere quem é novo.
spark.sql(\"\"\"
MERGE INTO governo_mg.{table_name} t
USING governo_mg.{staging_table} s
ON t.id_cidadao = s.id_cidadao
WHEN MATCHED THEN
  UPDATE SET
    t.nome = s.nome,
    t.valor_beneficio = s.valor_beneficio,
    t.data_pagamento = s.data_pagamento,
    t.status = s.status
WHEN NOT MATCHED THEN
  INSERT *
\"\"\")

spark.sql(\"\"\"
SELECT *
FROM governo_mg.{table_name}
ORDER BY id_cidadao
\"\"\").show(truncate=False)

print("UPDATE, DELETE e MERGE executados com sucesso.")
"""

submit_statement(livy_session_id, spark_code)

Statement submetido: 0
Estado do statement: waiting
Estado do statement: running
Estado do statement: running
Estado do statement: available
{
  "status": "error",
  "execution_count": 0,
  "ename": "AnalysisException",
  "evalue": "[INSERT_COLUMN_ARITY_MISMATCH.NOT_ENOUGH_DATA_COLUMNS] Cannot write to `spark_catalog`.`governo_mg`.`beneficios`, the reason is not enough data columns:\nTable columns: `id_cidadao`, `nome`, `valor_beneficio`, `data_pagamento`, `status`, `municipio`, `programa_social`.\nData columns: `id_cidadao`, `nome`, `valor_beneficio`, `data_pagamento`, `status`.",
  "traceback": [
    "Traceback (most recent call last):\n",
    "  File \"/opt/cloudera/parcels/CDH-7.3.2-1.cdh7.3.2.p0.77083870/lib/spark3/python/lib/pyspark.zip/pyspark/sql/session.py\", line 1631, in sql\n    return DataFrame(self._jsparkSession.sql(sqlQuery, litArgs), self)\n                     ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^\n",
    "  File \"/opt/cloudera/parcels/CDH-7.3.2-1.cdh7.3.2.p0.77

{'status': 'error',
 'execution_count': 0,
 'ename': 'AnalysisException',
 'evalue': '[INSERT_COLUMN_ARITY_MISMATCH.NOT_ENOUGH_DATA_COLUMNS] Cannot write to `spark_catalog`.`governo_mg`.`beneficios`, the reason is not enough data columns:\nTable columns: `id_cidadao`, `nome`, `valor_beneficio`, `data_pagamento`, `status`, `municipio`, `programa_social`.\nData columns: `id_cidadao`, `nome`, `valor_beneficio`, `data_pagamento`, `status`.',
 'traceback': ['Traceback (most recent call last):\n',
  '  File "/opt/cloudera/parcels/CDH-7.3.2-1.cdh7.3.2.p0.77083870/lib/spark3/python/lib/pyspark.zip/pyspark/sql/session.py", line 1631, in sql\n    return DataFrame(self._jsparkSession.sql(sqlQuery, litArgs), self)\n                     ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^\n',
  '  File "/opt/cloudera/parcels/CDH-7.3.2-1.cdh7.3.2.p0.77083870/lib/spark3/python/lib/py4j-0.10.9.7-src.zip/py4j/java_gateway.py", line 1322, in __call__\n    return_value = get_return_value(\n                   ^^^^^

In [ ]:
# Encerre a sessão ao final do notebook.
close_livy_session(livy_session_id)